# 10 Additional Preprocessing

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb009_sequential_aug.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb009_sequential_aug.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb009_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb009_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb009_sequential_aug.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2d3e675fb4fd24052bcebdf06cb5fb76dff380cb'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb009_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b8f6ffc8ea954268bac2b1620c3d3ca5d1f7bfe4'}}

## exp_cmb009_sequential_aug

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb009_sequential_aug.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb009_sequential_aug.py)  
Git blob: `2d3e675fb4fd24052bcebdf06cb5fb76dff380cb`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-009: frozen CMB-008 S6 batch128 plus one train-only geometry augmentation.

Arms: base, warp (balanced upper/lower +/-1px), rotate3, rotate5,
shear005, shear010, and a gated combo constructed AFTER separate comparisons.
No original/professor Test or new 438/480 used to train or select arms.

CLI 인자 (정적 추출): `--arm`, `--seed`, `--heldout`, `--selection`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb009_sequential_aug.py
#!/usr/bin/env python3
"""EXP-CMB-009: frozen CMB-008 S6 batch128 plus one train-only geometry augmentation.

Arms: base, warp (balanced upper/lower +/-1px), rotate3, rotate5,
shear005, shear010, and a gated combo constructed AFTER separate comparisons.
No original/professor Test or new 438/480 used to train or select arms.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,time,warnings
from pathlib import Path
import cv2
import joblib
import numpy as np
import sklearn
from sklearn.metrics import f1_score,confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from exp_cmb008_s5_early_stopping import (
    ROOT, WRITERS, LABELS, V3_SHA, HANDMADE_SHA,
    load_official, load_writers, PixelScaler255, prepareraw, process)
from stroke_normalization import topology

ARMS=("base","warp","rotate3","rotate5","shear005","shear010","combo")
OBSERVE=((7,9,"7_to_9"),(9,7,"9_to_7"),
         (13,0,"D_to_0"),(0,13,"0_to_D"),(11,8,"B_to_8"),(8,11,"8_to_B"))
BASE_SEED=20261118
FRACTION=.20
OFFICIAL=ROOT/"02_data/datasets/MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data/Hand_made"
YY,XX=np.mgrid[0:28,0:28].astype(np.float32)

def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()

def train_indices(y):
    """Class-balanced and predetermined before learning; never depends on OOF/Test."""
    selected=[]
    for c in range(16):
        ii=np.flatnonzero(y==c)
        order=np.random.default_rng(BASE_SEED+173*c).permutation(ii)
        selected.extend(order[:int(round(FRACTION*len(ii)))])
    ids=np.asarray(sorted(selected),dtype=np.int32)
    assert len(ids)==len(set(ids.tolist()))
    assert abs(len(ids)/len(y)-FRACTION)<.002
    return ids

def warp_map(kind,sign):
    # Continuous, zero-ended vertical window prevents hard row seams.
    if kind=="upper":
        weights=np.where(YY<=13,np.sin(np.pi*(YY+1)/15),0)
    else:
        weights=np.where(YY>=14,np.sin(np.pi*(28-YY)/15),0)
    return (XX-sign*weights).astype(np.float32),YY.astype(np.float32)

WARP_MAPS={(k,s):warp_map(k,s) for k in ("upper","lower") for s in (-1,1)}

def render(img,arm,index):
    """Same signs/upper-or-lower choice for every label; zero padding; linear sampling."""
    img=np.asarray(img,dtype=np.float32).reshape(28,28)
    bit=int((int(index)*1103515245+12345)&0x7fffffff)
    sign=1 if (bit%2) else -1
    if arm=="warp":
        region="upper" if (bit//2)%2==0 else "lower"
        mx,my=WARP_MAPS[region,sign]
        out=cv2.remap(img,mx,my,cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT,borderValue=0)
    elif arm.startswith("rotate"):
        angle=(3 if arm=="rotate3" else 5)*sign
        matrix=cv2.getRotationMatrix2D((13.5,13.5),angle,1.)
        out=cv2.warpAffine(img,matrix,(28,28),flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT,borderValue=0)
    elif arm.startswith("shear"):
        slope=(.05 if arm=="shear005" else .10)*sign
        matrix=np.asarray([[1.,slope,-13.5*slope],[0.,1.,0.]],dtype=np.float32)
        out=cv2.warpAffine(img,matrix,(28,28),flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT,borderValue=0)
    else:raise ValueError(arm)
    return out

def apply_aug(baseline,y,arm,combo_arms=None):
    if arm=="base":
        return baseline,{"arm":arm,"ratio":0,"attempted":0,"accepted":0,
                         "topology_rejected":0,"mass_rejected":0,"empty_rejected":0}
    operations=tuple(combo_arms if arm=="combo" else [arm])
    if not operations:raise ValueError("combo operations missing")
    ids=train_indices(y)
    orig=np.asarray(baseline[ids],dtype=np.float32).reshape(-1,28,28)
    edited=orig.copy()
    counts={"attempted":len(ids),"accepted":0,"topology_rejected":0,
            "mass_rejected":0,"empty_rejected":0}
    for pos,index in enumerate(ids):
        src=orig[pos]
        proposed=src.copy()
        for op in operations:proposed=render(proposed,op,index)
        if topology(proposed)!=topology(src):
            counts["topology_rejected"]+=1;continue
        mass=float(src.sum())
        if not (.7 <= float(proposed.sum())/max(mass,1.) <= 1.3):
            counts["mass_rejected"]+=1;continue
        if not np.any(proposed>=32):
            counts["empty_rejected"]+=1;continue
        edited[pos]=proposed
        counts["accepted"]+=1
    new=np.asarray(baseline,dtype=np.float32).copy()
    new[ids]=process(edited,"com",20)
    return new,{"arm":arm,"operations":list(operations),
                "ratio":float(len(ids)/len(y)),
                "official_aug_source_ids_sha256":hashlib.sha256(ids.tobytes()).hexdigest(),
                **counts}

def save_csv(path,rows):
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)

def measure(y,p):
    cm=confusion_matrix(y,p,labels=np.arange(16))
    return {"n":int(len(y)),"correct":int((y==p).sum()),
            "accuracy":float(np.mean(y==p)),
            "macro_f1":float(f1_score(y,p,labels=np.arange(16),average="macro",zero_division=0)),
            "pairs":{key:int(cm[a,b]) for a,b,key in OBSERVE},
            "class_errors":{LABELS[c]:int(cm[c].sum()-cm[c,c]) for c in range(16)},
            "class_support":{LABELS[c]:int(cm[c].sum()) for c in range(16)}}

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--arm",required=True,choices=ARMS)
    p.add_argument("--seed",required=True,type=int,choices=(42,43,44))
    p.add_argument("--heldout",required=True,type=int,choices=WRITERS)
    p.add_argument("--selection",type=Path)
    p.add_argument("--out",required=True,type=Path)
    a=p.parse_args()
    combo=None
    if a.arm=="combo":
        if not a.selection or not a.selection.is_file():raise ValueError("Need selection JSON")
        combo=json.loads(a.selection.read_text())["selected"]
        if len(combo)<2:
            print("COMBO_NOT_JUSTIFIED",json.dumps(combo),flush=True)
            a.out.mkdir(parents=True)
            (a.out/"skip.json").write_text(json.dumps({"reason":"fewer_than_two_gated_improvements","selected":combo}))
            return
    if a.out.exists():raise FileExistsError(a.out)
    a.out.mkdir(parents=True)
    started=time.perf_counter()
    official=load_official(OFFICIAL)
    handmade=load_writers(HANDMADE)
    assert official["sha256"]==V3_SHA
    assert all(handmade[w]["sha256"]==HANDMADE_SHA[w] for w in WRITERS)
    off_x,w_x,baseline_aug,_=prepareraw(official,handmade,"scale30_stroke28")
    off_x,delta_info=apply_aug(off_x,official["y_train"],a.arm,combo)
    folds=[w for w in WRITERS if w!=a.heldout]
    train_x=np.concatenate([off_x]+[w_x[w] for w in folds])
    train_y=np.concatenate([official["y_train"]]+[handmade[w]["y"] for w in folds])
    assert train_x.shape==(78720,784) and train_y.shape==(78720,)
    ho_x=w_x[a.heldout]
    ho_y=np.asarray(handmade[a.heldout]["y"],dtype=int)
    va_x=process(official["X_val"],"com",20)
    va_y=np.asarray(official["y_val"],dtype=int)
    mlp=MLPClassifier(hidden_layer_sizes=(256,),alpha=.0001,
        learning_rate_init=.001,solver="adam",activation="relu",
        early_stopping=False,batch_size=128,max_iter=200,
        n_iter_no_change=10,tol=.0001,random_state=a.seed)
    model=Pipeline([("scale_0_1",PixelScaler255()),("model",mlp)])
    with warnings.catch_warnings(record=True) as wrns:
        warnings.simplefilter("always")
        tick=time.perf_counter()
        model.fit(train_x,train_y)
        fit_seconds=time.perf_counter()-tick
    ho_pred=np.asarray(model.predict(ho_x),dtype=np.int32)
    va_pred=np.asarray(model.predict(va_x),dtype=np.int32)
    oof=measure(ho_y,ho_pred);val=measure(va_y,va_pred)
    save_csv(a.out/"oof_predictions.csv",
        [{"arm":a.arm,"seed":a.seed,"heldout":a.heldout,
          "index":int(i),"actual":LABELS[int(ho_y[i])],
          "pred":LABELS[int(ho_pred[i])]} for i in range(640)])
    from joblib import dump
    dump(model,a.out/"trained_model.joblib",compress=3)
    meta={"experiment":"EXP-CMB-009","arm":a.arm,"combo_operations":combo,
      "seed":a.seed,"heldout":a.heldout,"model_batch":128,
      "frozen":{"hidden_layer_sizes":[256],"alpha":.0001,
        "learning_rate_init":.001,"solver":"adam","activation":"relu",
        "early_stopping":False,"max_iter":200,"tol":.0001,"n_iter_no_change":10},
      "training_samples":len(train_y),"official_val_samples":len(va_y),
      "heldout_samples":len(ho_y),"official_sha256":official["sha256"],
      "writer_sha256":{str(w):handmade[w]["sha256"] for w in WRITERS},
      "baseline_aug":baseline_aug,"new_train_only_aug":delta_info,
      "original_test_used":False,"professor_test_used":False,
      "external_438_used_in_training_or_selection":False,
      "oof":oof,"official_val":val,"n_iter":int(mlp.n_iter_),
      "fit_seconds":float(fit_seconds),"elapsed_seconds":float(time.perf_counter()-started),
      "warnings":[str(w.message) for w in wrns],
      "sklearn_version":sklearn.__version__,
      "model_sha256":sha(a.out/"trained_model.joblib")}
    (a.out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n")
    print("CMB009_FIT_OK",json.dumps({"arm":a.arm,"seed":a.seed,"heldout":a.heldout,
      "oof":oof["accuracy"],"val":val["accuracy"],"7_to_9":oof["pairs"]["7_to_9"],
      "accepted":delta_info["accepted"],"fit_sec":round(fit_seconds,1)}),flush=True)
if __name__=="__main__":main()


## exp_cmb009_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb009_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb009_aggregate.py)  
Git blob: `b8f6ffc8ea954268bac2b1620c3d3ca5d1f7bfe4`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate exactly paired 12-fold CMB009 runs, gate arms, then assess gated combo.

CLI 인자 (정적 추출): `--incoming`, `--out`, `--phase`, `--selection`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb009_aggregate.py
#!/usr/bin/env python3
"""Aggregate exactly paired 12-fold CMB009 runs, gate arms, then assess gated combo."""
from __future__ import annotations
import argparse,csv,json,pathlib,statistics,hashlib
from collections import defaultdict

ARMS=("base","warp","rotate3","rotate5","shear005","shear010")
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
LABELS="0123456789ABCDEF"
WATCH=("7_to_9","9_to_7","D_to_0","0_to_D","B_to_8","8_to_B")

def save(path,data):
    path.write_text(json.dumps(data,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

def gather(directory,arms):
    metrics={}
    raw={}
    for arm in arms:
        for seed in SEEDS:
            for held in WRITERS:
                key=(arm,seed,held)
                d=directory/f"cmb009-{arm}-s{seed}-w{held}"
                if not d.exists():raise AssertionError(f"missing folder: {d}")
                m=json.loads((d/"manifest.json").read_text())
                assert (m["arm"],m["seed"],m["heldout"])==key
                assert m["training_samples"]==78720 and m["heldout_samples"]==640
                assert m["official_val_samples"]==19200 and m["model_batch"]==128
                assert not m["external_438_used_in_training_or_selection"]
                assert not m["original_test_used"] and not m["professor_test_used"]
                assert m["frozen"]["early_stopping"] is False
                assert m["sklearn_version"]=="1.8.0"
                assert m["new_train_only_aug"]["ratio"]==0 or abs(m["new_train_only_aug"]["ratio"]-.2)<.002
                assert hashlib.sha256((d/"trained_model.joblib").read_bytes()).hexdigest()==m["model_sha256"]
                with (d/"oof_predictions.csv").open(newline="",encoding="utf-8-sig") as f:
                    pp=list(csv.DictReader(f))
                assert len(pp)==640
                assert [int(x["index"]) for x in pp]==list(range(640))
                assert [x["actual"] for x in pp]==[LABELS[i%16] for i in range(640)]
                metrics[key]=m;raw[key]=pp
    for seed in SEEDS:
        for held in WRITERS:
            official={metrics[(a,seed,held)]["official_sha256"] for a in arms}
            hand={json.dumps(metrics[(a,seed,held)]["writer_sha256"],sort_keys=True) for a in arms}
            assert len(official)==len(hand)==1
    return metrics,raw

def summarize(metrics,arms):
    out={}
    for arm in arms:
        rows=[m for (a,_,_),m in metrics.items() if a==arm]
        def mean(k):
            return sum(r[k]["accuracy"] for r in rows)/len(rows)
        def macro(k):
            return sum(r[k]["macro_f1"] for r in rows)/len(rows)
        oc={}
        for pair in WATCH:
            oc[pair]=sum(r["oof"]["pairs"][pair] for r in rows)
        e={c:sum(r["oof"]["class_errors"][c] for r in rows) for c in LABELS}
        writer={str(w):sum(metrics[(arm,s,w)]["oof"]["accuracy"] for s in SEEDS)/3 for w in WRITERS}
        seeds={str(s):sum(metrics[(arm,s,w)]["oof"]["accuracy"] for w in WRITERS)/4 for s in SEEDS}
        out[arm]={"oof_accuracy":mean("oof"),"oof_macro_f1":macro("oof"),
            "official_val_accuracy":mean("official_val"),
            "official_val_macro_f1":macro("official_val"),
            "writer_oof":writer,"seed_oof":seeds,
            "oof_directional_counts":oc,"oof_class_errors":e,
            "oof_correct_sum":sum(r["oof"]["correct"] for r in rows),
            "oof_trials":7680,"official_val_trials":12*19200,
            "n_models":len(rows),
            "new_train_only_aug_accepted_mean":sum(r["new_train_only_aug"]["accepted"] for r in rows)/len(rows),
            "new_train_only_aug_attempted_mean":sum(r["new_train_only_aug"]["attempted"] for r in rows)/len(rows),
            "avg_epochs":sum(r["n_iter"] for r in rows)/len(rows)}
    return out

def paired(pred,arm):
    improved=broken=0
    for seed in SEEDS:
        for held in WRITERS:
            one=pred[("base",seed,held)]
            two=pred[(arm,seed,held)]
            assert len(one)==len(two)
            for a,b in zip(one,two):
                assert a["actual"]==b["actual"] and a["index"]==b["index"]
                was=a["actual"]==a["pred"];now=b["actual"]==b["pred"]
                improved+=int(not was and now)
                broken+=int(was and not now)
    return {"corrected":improved,"new_errors":broken,"net":improved-broken}

def evaluate(arms,summary,pred):
    ref=summary["base"]
    decisions={}
    for arm in arms:
        if arm=="base":continue
        x=summary[arm]
        val_drop_pp=100*(ref["official_val_accuracy"]-x["official_val_accuracy"])
        acc_gain_pp=100*(x["oof_accuracy"]-ref["oof_accuracy"])
        worst_writer_pp=min(100*(x["writer_oof"][w]-ref["writer_oof"][w]) for w in ref["writer_oof"])
        p=x["oof_directional_counts"];q=ref["oof_directional_counts"]
        focus_growth=sum(p[k]-q[k] for k in ("D_to_0","0_to_D","B_to_8","8_to_B"))
        # Predeclared exploratory gates. Do not treat these as validated independent-test thresholds.
        gates={"oof_gain_at_least_0p1pp":acc_gain_pp>=.10-1e-12,
               "official_val_drop_max_0p3pp":val_drop_pp<=.30+1e-12,
               "writer_worst_drop_max_1pp":worst_writer_pp>=-1.0-1e-12,
               "seven_to_nine_no_increase":p["7_to_9"]<=q["7_to_9"],
               "nine_to_seven_no_increase":p["9_to_7"]<=q["9_to_7"],
               "old_four_directions_net_growth_max_6":focus_growth<=6}
        decisions[arm]={"gates":gates,"pass":all(gates.values()),
          "oof_gain_pp":acc_gain_pp,"val_drop_pp":val_drop_pp,
          "worst_writer_delta_pp":worst_writer_pp,"old_four_count_delta":focus_growth,
          "paired_vs_baseline":paired(pred,arm)}
    return decisions

def main():
    a=argparse.ArgumentParser()
    a.add_argument("--incoming",required=True,type=pathlib.Path)
    a.add_argument("--out",required=True,type=pathlib.Path)
    a.add_argument("--phase",choices=("standalone","combo"),required=True)
    a.add_argument("--selection",type=pathlib.Path)
    x=a.parse_args()
    x.out.mkdir(parents=True,exist_ok=True)
    if x.phase=="standalone":
        metrics,pred=gather(x.incoming,ARMS)
        summary=summarize(metrics,ARMS)
        decision=evaluate(ARMS,summary,pred)
        selected=[]
        for armset in (("warp",),("rotate3","rotate5"),("shear005","shear010")):
            ok=[arm for arm in armset if decision[arm]["pass"]]
            if ok: selected.append(max(ok,key=lambda z:summary[z]["oof_accuracy"]))
        selection={"selected":selected,"criteria":"predeclared exploratory gates","combo_allowed":len(selected)>=2,
            "reason":"Only independently qualified arms combined; same 20 percent official cohort, composed transforms",
            "no_external438_or_original_professor_test_used":True}
        report={"experiment":"EXP-CMB-009","stage":"localwarp_rotation_shear",
            "frozen_batch_size":128,"n_models":72,"summary":summary,"decision":decision,"selection":selection}
        save(x.out/"standalone_summary.json",report)
        save(x.out/"selection.json",selection)
        with (x.out/"standalone_table.csv").open("w",newline="",encoding="utf-8-sig") as f:
            writer=csv.DictWriter(f,fieldnames=["arm","oof_accuracy","official_val_accuracy","oof_macro_f1","7_to_9","9_to_7","D_to_0","0_to_D","B_to_8","8_to_B","paired_corrected","paired_new_errors","paired_net","passes_gates"])
            writer.writeheader()
            for arm in ARMS:
                m=summary[arm];d=decision.get(arm,{})
                writer.writerow({"arm":arm,"oof_accuracy":m["oof_accuracy"],
                    "official_val_accuracy":m["official_val_accuracy"],
                    "oof_macro_f1":m["oof_macro_f1"],
                    **{k:m["oof_directional_counts"][k] for k in WATCH},
                    **{"paired_"+key:value for key,value in d.get("paired_vs_baseline",{}).items()},
                    "passes_gates":d.get("pass",True)})
        for group, names in (("1_Local_Warp",("warp",)),("2_Rotation",("rotate3","rotate5")),("3_Shear",("shear005","shear010"))):
            for name in names:
                s=summary[name];d=decision[name]
                print("STAGE_RESULT",json.dumps({"stage":group,"arm":name,
                     "oof_accuracy":s["oof_accuracy"],"official_val_accuracy":s["official_val_accuracy"],
                     "seven_to_nine":s["oof_directional_counts"]["7_to_9"],
                     "baseline_oof":summary["base"]["oof_accuracy"],
                     "gates":d["gates"],"approved_for_combo":d["pass"]}),flush=True)
        print("COMBO_SELECTION",json.dumps(selection),flush=True)
    else:
        if not x.selection:raise ValueError("must provide selection")
        sel=json.loads(x.selection.read_text())
        if not sel["combo_allowed"]:
            print("STAGE4_NO_COMBO",json.dumps(sel))
            save(x.out/"combination_summary.json",{"skipped":True,"selection":sel})
            return
        metrics,pred=gather(x.incoming,("base","combo"))
        summary=summarize(metrics,("base","combo"))
        decision=evaluate(("base","combo"),summary,pred)
        report={"experiment":"EXP-CMB-009","stage":"gated_combination",
                "selected_operations":sel["selected"],
                "summary":summary,"decision":decision,
                "combo_selected_as_research_candidate":decision["combo"]["pass"],
                "independent_test_used":False}
        save(x.out/"combination_summary.json",report)
        print("STAGE4_COMBO_RESULT",json.dumps(report),flush=True)
if __name__=="__main__":main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")